In [15]:
import numpy as np
import pandas as pd
from pyqlearning.q_learning import QLearning
import gymnasium as gym
from gymnasium import spaces
import pygame
import random
import copy

# Custom Maze Environment
class MazeEnv(gym.Env):
    def __init__(self):
        super(MazeEnv, self).__init__()
        # 10x10 maze: 0 = path, 1 = wall, 2 = goal
        self.maze = np.array([
            [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
            [1, 0, 1, 0, 0, 0, 1, 0, 0, 1],
            [1, 0, 1, 0, 1, 0, 1, 0, 1, 1],
            [1, 0, 0, 0, 1, 0, 0, 0, 0, 1],
            [1, 1, 1, 0, 1, 1, 1, 1, 0, 1],
            [1, 0, 0, 0, 0, 0, 0, 1, 0, 1],
            [1, 0, 1, 1, 1, 1, 0, 1, 0, 1],
            [1, 0, 1, 0, 0, 0, 0, 1, 0, 1],
            [1, 0, 0, 0, 1, 1, 0, 0, 0, 1],
            [1, 1, 1, 1, 1, 1, 1, 2, 1, 1]
        ])
        self.start_pos = (1, 1)
        self.goal_pos = (7, 7)
        self.current_pos = self.start_pos
        self.path = [self.start_pos]
        self.action_space = spaces.Discrete(4)  # 0: Up, 1: Down, 2: Left, 3: Right
        self.observation_space = spaces.Box(low=0, high=1, shape=(10, 10), dtype=np.int32)
        self.screen = None
        self.clock = None
        self.running = True

    def reset(self, seed=None):
        super().reset(seed=seed)
        self.current_pos = self.start_pos
        self.path = [self.start_pos]
        return np.copy(self.maze), {}

    def step(self, action):
        actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right
        next_pos = (self.current_pos[0] + actions[action][0], self.current_pos[1] + actions[action][1])
        if (0 <= next_pos[0] < self.maze.shape[0] and 
            0 <= next_pos[1] < self.maze.shape[1] and 
            self.maze[next_pos] != 1):
            self.current_pos = next_pos
        self.path.append(self.current_pos)

        reward = 1.0 if self.maze[self.current_pos] == 2 else -0.01
        done = self.maze[self.current_pos] == 2
        truncated = False
        info = {}
        return np.copy(self.maze), reward, done, truncated, info

    def render(self, mode='human', title="Maze", path=None):
        if self.screen is None:
            pygame.init()
            self.screen = pygame.display.set_mode((self.maze.shape[1] * 50, self.maze.shape[0] * 50))
            pygame.display.set_caption(title)
            self.clock = pygame.time.Clock()

        self.screen.fill((255, 255, 255))  # White background
        for i in range(self.maze.shape[0]):
            for j in range(self.maze.shape[1]):
                color = (0, 0, 0) if self.maze[i, j] == 1 else (255, 255, 255) if self.maze[i, j] == 0 else (0, 255, 0)
                pygame.draw.rect(self.screen, color, (j * 50, i * 50, 50, 50))
                if self.maze[i, j] == 2:
                    pygame.draw.rect(self.screen, (0, 255, 0), (j * 50, i * 50, 50, 50))

        # Draw path if provided, otherwise use current path
        if path:
            pygame.draw.lines(self.screen, (0, 0, 255), False, 
                             [(p[1] * 50 + 25, p[0] * 50 + 25) for p in path], 5)
            self.current_pos = path[-1]  # Update current position for agent rendering
        else:
            pygame.draw.lines(self.screen, (0, 0, 255), False, 
                             [(p[1] * 50 + 25, p[0] * 50 + 25) for p in self.path], 5)

        # Draw agent
        pygame.draw.circle(self.screen, (255, 0, 0), 
                          (self.current_pos[1] * 50 + 25, self.current_pos[0] * 50 + 25), 20)

        pygame.display.flip()
        self.clock.tick(20)
        for event in pygame.event.get():
            if event.type == pygame.QUIT or (event.type == pygame.KEYDOWN and event.key == pygame.K_q):
                self.running = False

    def close(self):
        if self.screen is not None:
            pygame.quit()

class MazeQLearning(QLearning):
    def __init__(self, env):
        super().__init__()
        self.q_df = pd.DataFrame(columns=['state', 'action', 'q_value'])
        self.env = env
        self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right
        self.action_map = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
        self.best_q_df = None
        self.best_reward = float('-inf')
        self.best_episode_path = None
        self.worst_reward = float('inf')
        self.worst_episode_path = None
        self.first_episode_path = None

    def get_state(self):
        return str(tuple(self.env.current_pos))

    def extract_possible_actions(self, state_key):
        pos = eval(state_key)
        possible_actions = []
        for i, action in enumerate(self.actions):
            next_pos = (pos[0] + action[0], pos[1] + action[1])
            if (0 <= next_pos[0] < self.env.maze.shape[0] and 
                0 <= next_pos[1] < self.env.maze.shape[1] and 
                self.env.maze[next_pos] != 1):
                possible_actions.append(i)
        return possible_actions

    def observe_reward_value(self, state_key, action_key):
        return self.env.step(action_key)[1]

    def select_action(self, state_key, next_action_list, q_df=None):
        if not next_action_list:
            return None
        if q_df is None:
            q_df = self.q_df
        if random.random() < self.epsilon_value:
            return random.choice(next_action_list)
        else:
            q_values = []
            for a in next_action_list:
                q_row = q_df[(q_df['state'] == state_key) & 
                            (q_df['action'] == str(a))]
                q_value = q_row['q_value'].iloc[0] if not q_row.empty else 0.0
                q_values.append(q_value)
            max_q = max(q_values)
            max_actions = [a for a, q in zip(next_action_list, q_values) if q == max_q]
            return random.choice(max_actions)

    def update_q(self, state_key, action_key, reward_value, next_state_key):
        next_action_list = self.extract_possible_actions(next_state_key)
        if next_action_list:
            next_q_values = []
            for a in next_action_list:
                q_row = self.q_df[(self.q_df['state'] == next_state_key) & 
                                 (self.q_df['action'] == str(a))]
                q_value = q_row['q_value'].iloc[0] if not q_row.empty else 0.0
                next_q_values.append(q_value)
            next_max_q = max(next_q_values)
        else:
            next_max_q = 0

        q_row = self.q_df[(self.q_df['state'] == state_key) & 
                         (self.q_df['action'] == str(action_key))]
        current_q = q_row['q_value'].iloc[0] if not q_row.empty else 0.0
        new_q = (1 - self.alpha_value) * current_q + self.alpha_value * (reward_value + self.gamma_value * next_max_q)

        if q_row.empty:
            new_row = pd.DataFrame({
                'state': [state_key],
                'action': [str(action_key)],
                'q_value': [new_q]
            })
            self.q_df = pd.concat([self.q_df, new_row], ignore_index=True)
        else:
            self.q_df.loc[q_row.index, 'q_value'] = new_q

    def learn(self, episodes=100):
        total_rewards = []
        for episode in range(episodes):
            obs, _ = self.env.reset()
            done = False
            episode_reward = 0
            steps = 0
            if episode == 0:
                self.first_episode_path = self.env.path.copy()

            while not done and steps < 200:  # Limit steps to avoid infinite loops
                state = self.get_state()
                action = self.select_action(state, self.extract_possible_actions(state))
                if action is None:
                    break
                obs, reward, done, truncated, info = self.env.step(action)
                next_state = self.get_state()
                self.update_q(state, action, reward, next_state)
                episode_reward += reward
                steps += 1
            total_rewards.append(episode_reward)
            if episode_reward > self.best_reward:
                self.best_reward = episode_reward
                self.best_q_df = self.q_df.copy()
                self.best_episode_path = self.env.path.copy()
            if episode_reward < self.worst_reward:
                self.worst_reward = episode_reward
                self.worst_episode_path = self.env.path.copy()
            print(f"Episode {episode + 1} completed. Reward: {episode_reward:.2f}, Steps: {steps}")

    def visualize_loop(self):
        self.env.reset()
        while self.env.running:
            # Visualize best episode
            self.env.render(title=f"Best Episode (Reward: {self.best_reward:.2f})", path=self.best_episode_path)
            pygame.time.wait(2000)  # Pause for 2 seconds
            # Visualize worst episode
            self.env.render(title=f"Worst Episode (Reward: {self.worst_reward:.2f})", path=self.worst_episode_path)
            pygame.time.wait(2000)  # Pause for 2 seconds
        self.env.close()

# Usage
env = MazeEnv()
q_learning = MazeQLearning(env)
q_learning.alpha_value = 0.1
q_learning.gamma_value = 0.9
q_learning.epsilon_value = 0.1

# Train the agent
q_learning.learn(episodes=100)

# Print learned Q-values (optional)
print("Final Q-Values:")
print(q_learning.q_df)

C:\Users\CL\AppData\Local\Temp\ipykernel_4692\1420282899.py:169: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.q_df = pd.concat([self.q_df, new_row], ignore_index=True)


Episode 1 completed. Reward: 0.25, Steps: 76
Episode 2 completed. Reward: -0.25, Steps: 126
Episode 3 completed. Reward: -0.19, Steps: 120
Episode 4 completed. Reward: -2.00, Steps: 200
Episode 5 completed. Reward: 0.53, Steps: 48
Episode 6 completed. Reward: -0.15, Steps: 116
Episode 7 completed. Reward: 0.05, Steps: 96
Episode 8 completed. Reward: 0.77, Steps: 24
Episode 9 completed. Reward: 0.49, Steps: 52
Episode 10 completed. Reward: 0.33, Steps: 68
Episode 11 completed. Reward: 0.09, Steps: 92
Episode 12 completed. Reward: 0.33, Steps: 68
Episode 13 completed. Reward: 0.01, Steps: 100
Episode 14 completed. Reward: 0.67, Steps: 34
Episode 15 completed. Reward: 0.55, Steps: 46
Episode 16 completed. Reward: 0.35, Steps: 66
Episode 17 completed. Reward: 0.57, Steps: 44
Episode 18 completed. Reward: 0.65, Steps: 36
Episode 19 completed. Reward: 0.57, Steps: 44
Episode 20 completed. Reward: 0.31, Steps: 70
Episode 21 completed. Reward: 0.57, Steps: 44
Episode 22 completed. Reward: 0.51

In [16]:
# Start visualization loop
print("Starting visualization loop. Press 'q' to quit.")
q_learning.visualize_loop()

Starting visualization loop. Press 'q' to quit.
